# Multivariate LSTM baseline for hourly PM2.5 (London, 2024)

This notebook implements a recurrent neural network that conditions one-hour-ahead predictions of fine particulate matter (`pm2_5`) on the previous 24 hours of all retained pollutant and meteorological proxies. The workflow complements the classical ARIMA baseline by allowing nonlinear interactions across inputs while preserving a strict temporal train–test protocol.

## Imports and project paths

Working directories may be either the repository root or `notebooks`; paths are resolved accordingly so artefacts (`data/`, `models/`) load without manual configuration.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential


def project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "london_2024.csv").exists():
        return cwd
    if (cwd.parent / "data" / "london_2024.csv").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find data/london_2024.csv. "
        "Run with cwd = thesis_2026 or thesis_2026/notebooks."
    )


PROJECT_ROOT = project_root()
DATA_PATH = PROJECT_ROOT / "data" / "london_2024.csv"
SCALER_PATH = PROJECT_ROOT / "models" / "lstm_scaler.pkl"
MODEL_PATH = PROJECT_ROOT / "models" / "lstm_model.keras"

LOOKBACK = 24
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Reproducibility: library versions and RNG seeds (copy output into thesis appendix if needed)
import random
import sys

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

try:
    import tensorflow as tf

    tf.random.set_seed(RNG_SEED)
except ImportError:
    pass

print("Python:", sys.version.split()[0])
for _name, _mod in (
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "sklearn"),
    ("statsmodels", "statsmodels"),
    ("tensorflow", "tensorflow"),
):
    try:
        _m = __import__(_mod)
        print(f"{_name}:", getattr(_m, "__version__", "?"))
    except ImportError:
        print(f"{_name}: (not installed)")
for _name, _mod in (("pmdarima", "pmdarima"), ("xgboost", "xgboost")):
    try:
        _m = __import__(_mod)
        print(f"{_name}:", getattr(_m, "__version__", "?"))
    except ImportError:
        print(f"{_name}: (not installed)")


## Data preparation and chronological split

Location identifiers are dropped because they are constant within this extract. The panel is sorted in time, forced to an hourly frequency, and missing values are forward-filled so that each timestamp carries the most recent observed multivariate state without using future observations. The sample is then partitioned into an initial training segment and a terminal test segment; no shuffle is applied, which mirrors real-time deployment where future data are unavailable at estimation time.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["time"])
df = df.drop(columns=["city", "latitude", "longitude"])
df = df.sort_values("time").set_index("time")
df = df.asfreq("h")
df = df.ffill()

feature_cols = list(df.columns)
pm25_idx = feature_cols.index("pm2_5")

split = int(len(df) * 0.8)
train_df = df.iloc[:split].copy()
test_df = df.iloc[split:].copy()
print(f"Train rows: {len(train_df)}, test rows: {len(test_df)}")
print(f"Train: {train_df.index.min()} → {train_df.index.max()}")
print(f"Test:  {test_df.index.min()} → {test_df.index.max()}")

## Min–max scaling without test-set leakage

Neural networks benefit from bounded, similarly scaled inputs. A `MinMaxScaler` maps each column to $[0, 1]$ using minimum and maximum statistics. Crucially, those statistics are estimated **only** on the training window; the held-out test segment is transformed with the training-derived extrema. Fitting the scaler on the full sample would couple train and test information and optimistic bias would enter both optimisation and evaluation. The fitted scaler is persisted so that any downstream application reproduces identical normalisation.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

SCALER_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, SCALER_PATH)
print(f"Saved scaler to {SCALER_PATH}")

## Sliding-window supervision and tensor shape

Each supervised example stacks **24 consecutive hourly vectors** of all scaled features as input. The regression target is **scaled `pm2_5` one hour after the end of that window** (one-step-ahead forecasting). Concretely, for time index $t$, the input tensor collects rows $t-23,\ldots,t$ and the label is the $pm2_5$ value at $t+1$. Training windows are drawn entirely from the training segment. For the test phase, the first 24 input hours that overlap the test calendar are taken from the **end of the training** period so that the earliest test prediction still respects causality; remaining windows advance hour by hour through the test segment. Stacking inputs yields the Keras three-dimensional layout `[samples, time steps, features]`.

In [ ]:
def make_sequences(data: np.ndarray, lookback: int, target_col: int) -> tuple[np.ndarray, np.ndarray]:
    """Past `lookback` rows (all features) -> pm2_5 at the next hour (scaled)."""
    X_list, y_list = [], []
    for t in range(lookback - 1, len(data) - 1):
        X_list.append(data[t - lookback + 1 : t + 1])
        y_list.append(data[t + 1, target_col])
    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32)


X_train, y_train = make_sequences(train_scaled, LOOKBACK, pm25_idx)
print(f"X_train shape (samples, timesteps, features): {X_train.shape}")

aug_test = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
X_test, y_test = make_sequences(aug_test, LOOKBACK, pm25_idx)
print(f"X_test shape: {X_test.shape}")

## Model architecture

Long short-term memory units maintain gated memory states suited to slowly varying environmental series. A dropout layer injects Bernoulli noise during training, which acts as a regulariser against overfitting to spurious temporal patterns. The readout is a single linear unit representing the (scaled) one-hour-ahead concentration, consistent with the scalar regression objective.

In [ ]:
n_features = X_train.shape[2]
model = Sequential(
    [
        LSTM(64, input_shape=(LOOKBACK, n_features), return_sequences=False),
        Dropout(0.2),
        Dense(1),
    ]
)
model.summary()

## Training configuration and optimisation

Parameters are updated with the Adam adaptive moment method using mean squared error as the loss; MSE emphasises quadratic penalties on large residuals, which is standard for continuous air-quality targets. A fixed epoch budget with moderate batch size balances convergence against computational cost for thesis experimentation. Batches are iterated in temporal order (`shuffle=False`) so that recurrent state patterns align with the underlying hourly process; a tail fraction of the training window serves as validation while preserving chronology.

In [ ]:
model.compile(optimizer="adam", loss="mse")
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=1,
)

## Test-set metrics on the original concentration scale

Network outputs live on the normalised scale of `pm2_5`. For interpretability, predictions and targets are mapped back to micrograms per cubic metre using the stored scaler inverse transform along the `pm2_5` column only. Root mean squared error and mean absolute error are then reported on the physical scale used in policy and health guidance.

In [ ]:
def inverse_pm25_column(scaled_1d: np.ndarray, scaler: MinMaxScaler, col_index: int) -> np.ndarray:
    n = len(scaled_1d)
    pad = np.zeros((n, scaler.n_features_in_), dtype=np.float64)
    pad[:, col_index] = np.asarray(scaled_1d).reshape(-1)
    inv = scaler.inverse_transform(pad)
    return inv[:, col_index]


y_pred_scaled = model.predict(X_test, verbose=0).reshape(-1)
y_test_orig = inverse_pm25_column(y_test, scaler, pm25_idx)
y_pred_orig = inverse_pm25_column(y_pred_scaled, scaler, pm25_idx)

mse = mean_squared_error(y_test_orig, y_pred_orig)
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test_orig, y_pred_orig))
print(f"Test RMSE (µg/m³): {rmse:.4f}")
print(f"Test MAE  (µg/m³): {mae:.4f}")

## Saving model metadata

A companion JSON file records the training timestamp, dataset version, architecture summary, and evaluation metrics alongside the model artifact.

In [ ]:
import json
from datetime import datetime, timezone

metadata = {
    "model": "lstm",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "data_version": str(DATA_PATH.name),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "architecture": {
        "lookback": LOOKBACK,
        "lstm_units": 64,
        "dropout": 0.2,
        "epochs": 20,
        "batch_size": 32,
    },
    "feature_columns": feature_cols,
    "metrics": {"rmse": round(rmse, 4), "mae": round(mae, 4)},
    "notes": "Multivariate LSTM, 24-hour lookback, one-step-ahead PM2.5.",
}
meta_path = MODEL_PATH.parent / "lstm_metadata.json"
with meta_path.open("w") as fh:
    json.dump(metadata, fh, indent=2)
print(f"Saved metadata to {meta_path}")
print(json.dumps(metadata, indent=2))